In [1]:
import pandas as pd
from pathlib import Path

airbnb = pd.read_csv("../../filtered_listings_reviews-ny.csv", low_memory=False)
airbnb.head()

,listing_id,review_id,listing_name,description,review,rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,neighborhood,price,availability_30,availability_60,availability_90,availability_365,availability_eoy
0,2992450,15066586,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,Large apartment nice kitchen and bathroom. Ken...,3.56,3.44,3.56,4.22,4.56,3.22,3.67,THIRD WARD,70.0,0,0,0,221,0
1,2992450,21810844,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,"This may be a little late, but just to say Ken...",3.56,3.44,3.56,4.22,4.56,3.22,3.67,THIRD WARD,70.0,0,0,0,221,0
2,2992450,28524578,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,Kenneth was ready when I got there and arrange...,3.56,3.44,3.56,4.22,4.56,3.22,3.67,THIRD WARD,70.0,0,0,0,221,0
3,2992450,35913434,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,We were pleased to see how 2nd Street and the ...,3.56,3.44,3.56,4.22,4.56,3.22,3.67,THIRD WARD,70.0,0,0,0,221,0
4,2992450,38893053,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,"The flat is not in a good area, while we were ...",3.56,3.44,3.56,4.22,4.56,3.22,3.67,THIRD WARD,70.0,0,0,0,221,0


In [2]:
import numpy as np

def load_glove(path, dim):
    vectors = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            word = parts[0]
            vals = np.asarray(parts[1:], dtype=np.float32)
            if vals.shape[0] != dim:
                continue
            vectors[word] = vals
    return vectors

glove = load_glove("../../glove.6B/glove.6B.100d.txt", dim=100)


In [3]:
import re

token_re = re.compile(r"[a-z]+(?:'[a-z]+)?")

def text_to_glove_avg(text, glove, dim):
    
    tokens = token_re.findall(text.lower())
    vecs = [glove[t] for t in tokens if t in glove]

    if not vecs:
        return np.zeros(dim, dtype=np.float32)

    return np.mean(vecs, axis=0).astype(np.float32)


In [4]:
DIM = 100

airbnb["desc_emb"] = airbnb["description"].apply(lambda x: text_to_glove_avg(x, glove, DIM))
airbnb["review_emb"] = airbnb["review"].apply(lambda x: text_to_glove_avg(x, glove, DIM))
airbnb.head()


,listing_id,review_id,listing_name,description,review,rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,...,review_scores_value,neighborhood,price,availability_30,availability_60,availability_90,availability_365,availability_eoy,desc_emb,review_emb
0,2992450,15066586,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,Large apartment nice kitchen and bathroom. Ken...,3.56,3.44,3.56,4.22,4.56,...,3.67,THIRD WARD,70.0,0,0,0,221,0,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.052746892, 0.16154227, 0.37204847, -0.2111..."
1,2992450,21810844,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,"This may be a little late, but just to say Ken...",3.56,3.44,3.56,4.22,4.56,...,3.67,THIRD WARD,70.0,0,0,0,221,0,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.1777499, 0.16362435, 0.34134158, -0.170131..."
2,2992450,28524578,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,Kenneth was ready when I got there and arrange...,3.56,3.44,3.56,4.22,4.56,...,3.67,THIRD WARD,70.0,0,0,0,221,0,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.04268333, 0.11577168, 0.35954562, -0.23441..."
3,2992450,35913434,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,We were pleased to see how 2nd Street and the ...,3.56,3.44,3.56,4.22,4.56,...,3.67,THIRD WARD,70.0,0,0,0,221,0,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.101399995, 0.16551188, 0.31988055, -0.2034..."
4,2992450,38893053,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,"The flat is not in a good area, while we were ...",3.56,3.44,3.56,4.22,4.56,...,3.67,THIRD WARD,70.0,0,0,0,221,0,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.17374961, 0.17287166, 0.40169567, -0.28471..."


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Fit TF-IDF on BOTH fields so weights are meaningful
corpus = pd.concat([airbnb["description"].fillna(""), airbnb["review"].fillna("")], axis=0).astype(str)

tfidf = TfidfVectorizer(lowercase=True, token_pattern=r"[a-z]+(?:'[a-z]+)?", min_df=2)
tfidf.fit(corpus)

vocab = tfidf.vocabulary_
idf = tfidf.idf_

def text_to_glove_tfidf(text, glove, dim, vocab, idf):

    tokens = token_re.findall(text.lower())
    num = np.zeros(dim, dtype=np.float32)
    den = 0.0

    for t in tokens:
        if t in glove and t in vocab:
            w = float(idf[vocab[t]])  # IDF weight
            num += w * glove[t]
            den += w

    if den == 0.0:
        return np.zeros(dim, dtype=np.float32)
    return (num / den).astype(np.float32)


In [6]:
DIM = 100

airbnb["desc_emb_tfidf"] = airbnb["description"].apply(lambda x: text_to_glove_tfidf(x, glove, DIM, vocab, idf))
airbnb["review_emb_tfidf"] = airbnb["review"].apply(lambda x: text_to_glove_tfidf(x, glove, DIM, vocab, idf))
airbnb.head()

,listing_id,review_id,listing_name,description,review,rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,...,price,availability_30,availability_60,availability_90,availability_365,availability_eoy,desc_emb,review_emb,desc_emb_tfidf,review_emb_tfidf
0,2992450,15066586,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,Large apartment nice kitchen and bathroom. Ken...,3.56,3.44,3.56,4.22,4.56,...,70.0,0,0,0,221,0,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.052746892, 0.16154227, 0.37204847, -0.2111...","[-0.07131217, 0.20671527, 0.1649757, 0.0025740...","[-0.024323659, 0.17775345, 0.33285615, -0.1831..."
1,2992450,21810844,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,"This may be a little late, but just to say Ken...",3.56,3.44,3.56,4.22,4.56,...,70.0,0,0,0,221,0,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.1777499, 0.16362435, 0.34134158, -0.170131...","[-0.07131217, 0.20671527, 0.1649757, 0.0025740...","[-0.16767982, 0.17152984, 0.2768583, -0.097529..."
2,2992450,28524578,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,Kenneth was ready when I got there and arrange...,3.56,3.44,3.56,4.22,4.56,...,70.0,0,0,0,221,0,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.04268333, 0.11577168, 0.35954562, -0.23441...","[-0.07131217, 0.20671527, 0.1649757, 0.0025740...","[-0.057433996, 0.12556776, 0.32172346, -0.2076..."
3,2992450,35913434,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,We were pleased to see how 2nd Street and the ...,3.56,3.44,3.56,4.22,4.56,...,70.0,0,0,0,221,0,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.101399995, 0.16551188, 0.31988055, -0.2034...","[-0.07131217, 0.20671527, 0.1649757, 0.0025740...","[-0.09388068, 0.19623612, 0.26474038, -0.16637..."
4,2992450,38893053,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,"The flat is not in a good area, while we were ...",3.56,3.44,3.56,4.22,4.56,...,70.0,0,0,0,221,0,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.17374961, 0.17287166, 0.40169567, -0.28471...","[-0.07131217, 0.20671527, 0.1649757, 0.0025740...","[-0.1542468, 0.1489564, 0.35990825, -0.2933897..."


In [7]:
airbnb.to_csv("airbnb_glove_embeddings-ny.csv", index=False)